## Quantum Simulation & Linear Algebra

This branch form the strongest between [quantum computing](https://en.wikipedia.org/wiki/Quantum_computing) and [High-Performance Computing (HPC)](https://en.wikipedia.org/wiki/High-performance_computing) environments. While traditional supercomputers spend vast CPU/GPU node-hours solving large-scale systems, [partial differential equations (PDEs)](https://en.wikipedia.org/wiki/Partial_differential_equation), and quantum state dyanmicas via finite-element or [tensor network approximations](https://en.wikipedia.org/wiki/Tensor_network), quantum processors map these [Hilber space](https://en.wikipedia.org/wiki/Hilbert_space) representations directly onto qubit operations.

In modern hybrid [HPC-QC](https://www.hpcqc.org/home) workflows (e.g., using [CUDA-Q](https://developer.nvidia.com/cuda-q) or [OpenMPI/MPI](https://www.open-mpi.org/) quantum runtimes), classical supercomputing clusters manage heavy data pre-processing, [matrix partitioning](https://en.wikipedia.org/wiki/Block_matrix), [boundary conditions](https://en.wikipedia.org/wiki/Boundary_value_problem), classical error tracking, while delegating exponential matrix transformations directly to the quantum processing unit(QPU).

### 1. [Hamiltonian Simulation](https://en.wikipedia.org/wiki/Hamiltonian_simulation) (Trotterization, LCU, QSP)

* **Problem Domain:** [Quantum Physics](https://en.wikipedia.org/wiki/Quantum_mechanics), [Condensed Matter Physics](https://en.wikipedia.org/wiki/Condensed_matter_physics), [Material Science](https://en.wikipedia.org/wiki/Materials_science), [Quantum Chemistry](https://en.wikipedia.org/wiki/Quantum_chemistry).

* **Core Function:** Simulates the continuous time evolution operator $U(t) = e^{-iHt}$ of a physical system governed by a Hamiltonian $H$.

* **HPC Mapping & Convergence:** Classical supercomputers struggle with Hamiltonian simulation because storing state vectors requires memomry scaling exponentially as $\mathcal{O}(2^n)$. Quantum devices use $n$ qubits to hold these vectors, while HPC systems calcualte time-step intervals and operator decompositions (e.g., [Suzuki-Trotter](https://en.wikipedia.org/wiki/Time-evolving_block_decimation) products or [Linear Combinations of Unitaries](https://pennylane.ai/demos/tutorial_lcu_blockencoding)).

* **Algorithmic Mechanism:**

    1. The continuous system Hamiltonian is decomposed into local Pauli terms ($H = \sum_k H_k$).

    2. Time t is discretized into small intervals $\Delta t$ using [Lie-Trotter](https://en.wikipedia.org/wiki/Lie_product_formula)-Suzuki product formulas or [Quantum Signal Processing (QSP)](https://arxiv.org/pdf/2308.01501):
    $$
        e^{-iHt} \approx \Big( \prod_k e^{-iH_k \Delta t} \Big)^r
    $$

    3. Quantum circuits execute these product steps squentially to evolve state probabilities.

* **Complexity:**

    * **Quantum:** $\mathcal{O}(poly(t, \log{N}, 1/\epsilon))$. 

    * **Classical/HPC:** Exact exponential requires $\mathcal{O}(N^3) = \mathcal{O}(2^{3n})$ oprations.

#### Example: Hamilton Simulation (Trotterization)

Simulating $U(t) = e^{-iHt}$ for a 1D Ising chain with Hamiltonian $H = Z_0 Z_1 + X_0 + X_1$ using a 1st-order Trotter-Suzuki decomposition.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# Simulation parameters
t = Parameter('t')  # Time parameter
trotter_steps = 2
dt = t / trotter_steps

# Create circuit for 2 qubits
qc = QuantumCircuit(2)

for _ in range(trotter_steps):
    # 1. ZZ-interaction: exp(-i * dt * Z_0 Z_1)
    qc.rzz(2 * dt, 0, 1)
    
    # 2. X-field terms: exp(-i * dt * X_0) and exp(-i * dt * X_1)
    qc.rx(2 * dt, 0)
    qc.rx(2 * dt, 1)

    # 3. Adding barrier for visual convenience
    qc.barrier()

# Inspect the Trotterized circuit assigned for t = 1.0
bound_qc = qc.assign_parameters({t: 1.0})
print("Hamiltonian Simulation Circuit (Trotter Step = 2):")
print(bound_qc.draw('text'))

Hamiltonian Simulation Circuit (Trotter Step = 2):
             ┌───────┐ ░         ┌───────┐ ░ 
q_0: ─■──────┤ Rx(1) ├─░──■──────┤ Rx(1) ├─░─
      │ZZ(1) ├───────┤ ░  │ZZ(1) ├───────┤ ░ 
q_1: ─■──────┤ Rx(1) ├─░──■──────┤ Rx(1) ├─░─
             └───────┘ ░         └───────┘ ░ 


### 2. [HHL Algorithm (Harrow–Hassidim–Lloyd)](https://en.wikipedia.org/wiki/HHL_algorithm)

* **Problem Domain:** Systems of [Linear Equations](https://en.wikipedia.org/wiki/Linear_equation) ($Ax = b$), [Fluid Dynamics (Navier-Stokes)](https://en.wikipedia.org/wiki/Fluid_dynamics), [Structural Engineering](https://en.wikipedia.org/wiki/Structural_engineering), [Financial Modeling](https://en.wikipedia.org/wiki/Financial_modeling).

* **Core Function:** Solves $Ax = b$ for large, sparse $N \times N$ matrix A, outputting a quantum state $\ket{x}$ proportional to solution vector.

* **HPC Mapping & Convergence:** Large-scale linear system solvers are the backbone of classical HPC workloads (e.g., [Conjugate Gradient](https://en.wikipedia.org/wiki/Conjugate_gradient_method), [Krylov subspace](https://en.wikipedia.org/wiki/Krylov_subspace) methods). HHL accelerates these task exponentially relative to system dimension $N$, allowing HPC orchestrators to use QPUs for high-dimensional matrix inversions.

* **Algorithmic Mechanism:**
    1. **State Preparation:** Loads vector $b$ into quantum state $\ket{b} = \sum \beta_j \ket{u_j}$.

    2. **[Quantum Phase Estimation (QPE)](https://en.wikipedia.org/wiki/Quantum_phase_estimation_algorithm):** Applies $e^{iAt}$ to map matrix eigenvalues $\lambda_j$ of $A$ into a target phase register.

    3. **Controlled Rotation:** Performs a controlled rotation using an ancilla qubit to achieve propotional inversion $\lambda_j^{-1}$.
    
    4. **Uncomputation / Inversion QPE:** Undoes the phase estimation step isolating state     $\ket{x} \propto \sum_{\lambda_j}^{\beta_j} \ket{u_j}$.

* **Complexity:**
    * **Quantum:** $\mathcal{O}(s^2 \kappa^2 \log{N}/\epsilon)$ (where $s$ is sparsity and $\kappa$ is the condition number).

    * **Classical/HPC:** Conjugate Gradient Method takes $\mathcal{O}(N s\kappa \log{1/\epsilon}))$.

#### Example: HHL Algorithm (Linear Solver Workflow $Ax = b$)
Demonstrating the four programmatic steps of the HHL pipeline to solve $Ax = b$ using Qiskit primitives.

In [2]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT, RYGate

# Registers: Ancilla (1), Phase/Eigenvalue (2), System/b (1)
ancilla = QuantumRegister(1, name='ancilla')
clock = QuantumRegister(2, name='clock')
system = QuantumRegister(1, name='system')
c_out = ClassicalRegister(1, name='c_out')

qc = QuantumCircuit(ancilla, clock, system, c_out)

# Step 1: State Preparation |b>
qc.x(system[0])  # Encode state |1> as |b>

# Step 2: Quantum Phase Estimation (QPE) - Phase Register Superposition
qc.h(clock)
# (In practice: Controlled Unitaries C-e^{iAt} act here on system conditioned on clock)

# Inverse QFT on clock register
qc.append(QFT(num_qubits=2, inverse=True).to_gate(), clock)

# Step 3: Controlled Rotation (Reciprocal Phase Inversion ~ 1/lambda)
qc.append(RYGate(np.pi / 2).control(2), [*clock, ancilla[0]])

# Step 4: Uncompute QPE
qc.append(QFT(num_qubits=2, inverse=False).to_gate(), clock)

# Measure ancilla qubit (success flag)
qc.measure(ancilla, c_out)

print("\nConceptual HHL Circuit Architecture:")
print(qc.draw('text'))


Conceptual HHL Circuit Architecture:
                       ┌─────────┐        ┌─┐
ancilla: ──────────────┤ Ry(π/2) ├────────┤M├
         ┌───┐┌───────┐└────┬────┘┌──────┐└╥┘
clock_0: ┤ H ├┤0      ├─────■─────┤0     ├─╫─
         ├───┤│  IQFT │     │     │  QFT │ ║ 
clock_1: ┤ H ├┤1      ├─────■─────┤1     ├─╫─
         ├───┤└───────┘           └──────┘ ║ 
 system: ┤ X ├─────────────────────────────╫─
         └───┘                             ║ 
c_out: 1/══════════════════════════════════╩═
                                           0 


C:\Users\gandh\AppData\Local\Temp\ipykernel_17592\3920476162.py:21: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(num_qubits=2, inverse=True).to_gate(), clock)
C:\Users\gandh\AppData\Local\Temp\ipykernel_17592\3920476162.py:27: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(num_qubits=2, inverse=False).to_gate(), clock)


### 3. [Quantum Singular Value Transformation (QSVT)](https://en.wikipedia.org/wiki/Quantum_singular_value_transformation)

* **Pronlem Domain:** [Matrix inversion](https://en.wikipedia.org/wiki/Invertible_matrix), [Differential Equations](https://en.wikipedia.org/wiki/Differential_equation), [Quantum Machine Learning](https://en.wikipedia.org/wiki/Quantum_machine_learning), [Signal Processing](https://en.wikipedia.org/wiki/Signal_processing).

* **Core Function:** Unifies linear algebra algorithms under a single framework by applying [polynomial transformations](https://en.wikipedia.org/wiki/Polynomial_transformation) to the singular values of a block-encoded matrix.

* **HPC Mapping & Convergence:** Serves as the ultimate generalized runtime component for HPC-quantum linear algebra libraries. Instead of building bespoke circuits for simulation or matrix inversion, QSVT allows an HPC system to specify arbitrary target functions $P(x)$ applied directly to system matrices.

* **Algorithmic Mechanism:**
    1. Embeds an arbitrary non-unitary matrix $A$ into a higher-dimensional unitary matrix $U$ using [Block Encoding](https://pennylane.ai/demos/tutorial_block_encoding).

    2. Applies single-qubit phase rotations interleaved with calls $U$ and $U^\dagger$.

    3. Transforms matrix singular values $\sigma_i$ into polynomial values $P(\sigma_i)$ via quantum signal processing.

* **Complexity:**
    * **Quantum:** Optimal logarithmic gate scaling in error $\mathcal{\log{1/\epsilon}}$.

| Dimension | Classical Supercomputing (HPC) | Quantum Processing Unit (QPU) |
| :--- | :--- | :--- |
| **Vector Space Storage** | Explicit array allocation scaling as $\mathcal{O}(2^n)$ | Hilbert Space superposition scaling as $n$ qubits |
| **Linear Solvers ($Ax=b$)** | Iterative algorithms bound linearly by size $N$ | Logarithmic scaling with matrix dimension $\mathcal{O}(\log N)$ |
| **Data Extraction** | Full access to every element in output vector $x$ | Global statistical properties / expectation values $\langle x \vert M \vert x \rangle$ |

#### Example: Quantum Singular Value Transformation (QSVT)
Constructing a 1D QSVT transformation block over a block-encoded scalar/matrix $A$ using Phase Shift Factorization.

In [3]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import RZGate

def qsvt_block_1d(phase_angles):
    """
    Constructs a QSVT sequence given an array of phase angles.
    """
    qc = QuantumCircuit(2, name="QSVT")
    
    for i, phi in enumerate(phase_angles):
        # 1. Projector-controlled phase shift (Phase Gate on Ancilla)
        qc.rz(2 * phi, 0)
        
        # 2. Block Encoding U(A) operation (Simulated via CZ / Rotation)
        qc.ch(0, 1)
        qc.cz(0, 1)
        
        # Apply inverse block encoding on odd steps
        if i % 2 == 1:
            qc.ch(0, 1)

    return qc

# Example: 3-step polynomial transformation
angles = [0.456, -1.231, 0.785]
qsvt_circuit = qsvt_block_1d(angles)

print("\nQSVT Sequence Circuit Layout:")
print(qsvt_circuit.draw('text'))


QSVT Sequence Circuit Layout:
     ┌───────────┐        ┌────────────┐             ┌──────────┐        
q_0: ┤ Rz(0.912) ├──■───■─┤ Rz(-2.462) ├──■───■───■──┤ Rz(1.57) ├──■───■─
     └───────────┘┌─┴─┐ │ └────────────┘┌─┴─┐ │ ┌─┴─┐└──────────┘┌─┴─┐ │ 
q_1: ─────────────┤ H ├─■───────────────┤ H ├─■─┤ H ├────────────┤ H ├─■─
                  └───┘                 └───┘   └───┘            └───┘   
